In [9]:
#实现一个上周的小市值策略的回测 12月15号 到 12月19号
#加载数据
import sys,pandas,numpy,os
from pathlib import Path

In [ ]:
#以下是linux环境下的路径
project_path = Path("/home/lty/codes_claude/quant_backtest_claude/examples/self_test.ipynb").parent.parent

In [17]:
#以下是windows环境下的路径
project_path = Path("d:/360MoveData/Users/Lenovo/Desktop/cuhksz/quant_backtest_claude/examples/self_test.ipynb").parent.parent

In [18]:
#将项目路径插入系统寻找路径首位
#project_path = Path.cwd().parent.parent
print(str(project_path))
sys.path.insert(0,str(project_path))

d:\360MoveData\Users\Lenovo\Desktop\cuhksz\quant_backtest_claude


In [19]:
#导入数据处理模块的类
#查看当前目录
os.chdir(str(project_path))
print(os.getcwd())
from data.loader import TushareDataLoader,DataManager

d:\360MoveData\Users\Lenovo\Desktop\cuhksz\quant_backtest_claude


In [20]:
#导入流通市值矩阵
dm = DataManager()
dm.load('20250101','20251219')
market_cap = dm.market_cap

2026-02-13 18:04:15,367 - INFO - TushareDataLoader 初始化完成，数据目录: d:\360MoveData\Users\Lenovo\Desktop\cuhksz\quant_backtest_claude\storage
2026-02-13 18:04:18,454 - INFO - 加载日线数据: 13606806 条, 5739 只股票, 3916 个交易日
2026-02-13 18:04:22,234 - INFO - 加载每日指标: 13515700 条
2026-02-13 18:04:22,893 - INFO - DataManager 加载完成: 20250101 ~ 20251219
2026-02-13 18:04:22,963 - INFO -   日线: 1270277 条, 5490 只股票


In [21]:
print(market_cap)

ts_code        000001.SZ     000002.SZ    000004.SZ     000006.SZ  \
trade_date                                                          
2025-01-02  2.218096e+07  8.482734e+06  187715.2399  9.787464e+05   
2025-01-03  2.208393e+07  8.351497e+06  169446.7610  9.328466e+05   
2025-01-06  2.220037e+07  8.327635e+06  165740.1131  9.044967e+05   
2025-01-07  2.233621e+07  8.411150e+06  173550.5497  9.247466e+05   
2025-01-08  2.231681e+07  8.303774e+06  177389.5779  9.395966e+05   
...                  ...           ...          ...           ...   
2025-12-15  2.233621e+07  5.810256e+06  143235.4651  1.325695e+06   
2025-12-16  2.227799e+07  5.929563e+06  138072.6341  1.263595e+06   
2025-12-17  2.237502e+07  5.917632e+06  135027.8876  1.286545e+06   
2025-12-18  2.258849e+07  5.810256e+06  138072.6341  1.289245e+06   
2025-12-19  2.254968e+07  5.846048e+06  139925.9581  1.320295e+06   

ts_code       000007.SZ    000008.SZ     000009.SZ    000010.SZ    000011.SZ  \
trade_date            

In [22]:
from strategy.base import BaseStrategy,StrategyConfig
from strategy.small_cap import SmallCapStrategy

In [23]:
print("\n" + "="*60)
print("策略测试")
print("="*60)

results = {}

# 1. 纯小市值策略
print("\n【1. 纯小市值策略】")
strategy1 = SmallCapStrategy(
    n_stocks=10,
    rebalance_freq="weekly",
    max_market_cap=None,  # 不做市值限制，因为是模拟数据
)
weights1 = strategy1.run(dm)
print(weights1)

2026-02-13 18:04:34,470 - INFO - 运行策略: small_cap



策略测试

【1. 纯小市值策略】


2026-02-13 18:04:35,178 - INFO - 策略运行完成，生成 235 个交易日的权重


ts_code     000001.SZ  000002.SZ  000004.SZ  000006.SZ  000007.SZ  000008.SZ  \
trade_date                                                                     
2025-01-02        0.0        0.0        0.0        0.0        0.0        0.0   
2025-01-03        0.0        0.0        0.0        0.0        0.0        0.0   
2025-01-06        0.0        0.0        0.0        0.0        0.0        0.0   
2025-01-07        0.0        0.0        0.0        0.0        0.0        0.0   
2025-01-08        0.0        0.0        0.0        0.0        0.0        0.0   
...               ...        ...        ...        ...        ...        ...   
2025-12-15        0.0        0.0        0.0        0.0        0.0        0.0   
2025-12-16        0.0        0.0        0.0        0.0        0.0        0.0   
2025-12-17        0.0        0.0        0.0        0.0        0.0        0.0   
2025-12-18        0.0        0.0        0.0        0.0        0.0        0.0   
2025-12-19        0.0        0.0        

In [24]:
print(f"权重矩阵形状: {weights1.shape}")
print(f"平均持股数: {(weights1 > 0).sum(axis=1).mean():.1f}")
print(f"平均换手率: {strategy1.get_turnover().mean():.2%}")

权重矩阵形状: (235, 5490)
平均持股数: 9.9
平均换手率: 3.40%


In [25]:
price = dm.price
returns = price.pct_change().fillna(0)
portfolio_returns = (weights1.shift(1) * returns).sum(axis=1)
cum_returns = (1 + portfolio_returns).cumprod() - 1

In [26]:
#portfolio_returns
cum_returns

trade_date
2025-01-02    0.000000
2025-01-03    0.000000
2025-01-06    0.000000
2025-01-07    0.008242
2025-01-08    0.046459
                ...   
2025-12-15    1.042141
2025-12-16    1.059533
2025-12-17    1.028424
2025-12-18    1.008411
2025-12-19    0.988015
Length: 235, dtype: float64

In [27]:
from backtest.engine import BacktestResult,BacktestConfig,BacktestEngine,BacktestResult
from backtest.report import BacktestReport

In [28]:
# 3. 执行回测
print("\n【3. 执行回测】")

# 配置回测参数
config = BacktestConfig(
    commission_rate=0.0003,
    slippage=0.001,
    stamp_duty=0.001,
    trade_delay=1,
    apply_limit_rules=True,
    apply_suspend_rules=True,
    initial_capital=1000000,
)

engine = BacktestEngine(config)
result = engine.run(weights1, dm)
report = BacktestReport(result, '小市值')
report.print_summary()

2026-02-13 18:04:43,825 - INFO - 开始回测...
2026-02-13 18:04:43,842 - INFO - 回测区间: 2025-01-02 00:00:00 ~ 2025-12-19 00:00:00
2026-02-13 18:04:43,844 - INFO - 交易日数: 235, 股票数: 5490



【3. 执行回测】


2026-02-13 18:04:44,334 - INFO - 回测完成！总收益: 66.97%



回测报告 - 小市值

回测区间: 2025-01-02 ~ 2025-12-19
交易天数: 235

【收益指标】
  累计收益率: 66.97%
  年化收益率: 73.28%

【风险指标】
  年化波动率: 40.28%
  最大回撤:   22.79%
  95% VaR:    2.94%

【风险调整收益】
  夏普比率:   1.74
  卡尔玛比率: 3.21
  索提诺比率: 2.17

【交易指标】
  胜率:       50.64%
  盈亏比:     1.28
  平均换手率: 6.93%

